In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import PatchCollection, LineCollection
from matplotlib.colors import Normalize
from matplotlib import cm
from matplotlib.patches import Polygon

# 1. SETUP FONTS & SIZING
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm'

# INCREASE FONT SIZE relative to the figure
plt.rcParams['font.size'] = 14       # Base text size
plt.rcParams['axes.labelsize'] = 16  # Axis labels
plt.rcParams['xtick.labelsize'] = 12 # Ticks

# 2. DEFINE PHYSICS FUNCTIONS
def ne_k(nmu, delta, k):
    return 2 * np.sqrt((nmu + 2 * np.cos(k))**2 + 4 * delta**2 * np.sin(k)**2)

def ks3(nmu, delta):
    # This formula is valid for both Delta < 1 and Delta > 1
    # For Delta=2, (Delta^2 - 1) = 3, so argument is nmu/6.
    # Since nmu is [-3,3], argument is [-0.5, 0.5], which is valid for arccos.
    return np.arccos(nmu / (2 * (delta**2 - 1)))

# 3. PLOTTING HELPER FUNCTION
def plot_band_panel(ax, delta, nmu):
    cmap = plt.get_cmap('viridis')

    # --- CALCULATE CURVE DATA (ks3) ---
    # We calculate this for ALL Deltas now, as the curve always exists.
    ks3_values = ks3(nmu, delta)
    valid_indices = ~np.isnan(ks3_values) & ~np.isinf(ks3_values)
    nmu_valid = nmu[valid_indices]
    ks3_valid = ks3_values[valid_indices]

    # --- BACKGROUND LOGIC ---
    if delta > 1.0:
        # CASE A: Discrete Background (Purple | Yellow)
        # Split at mu = 0
        ax.axvspan(xmin=nmu.min(), xmax=0, color=cmap(0), alpha=0.25)
        ax.axvspan(xmin=0, xmax=nmu.max(), color=cmap(1.0), alpha=0.25)

    else:
        # CASE B: Gradient Background (Delta < 1)
        # Boundary at +/- 2(Delta^2 - 1)
        boundary_val = abs(2 * (delta**2 - 1))

        # Grey dashed lines
        ax.axvline(x=-boundary_val, color='grey', linestyle='--', alpha=1.0, linewidth=0.8)
        ax.axvline(x=boundary_val, color='grey', linestyle='--', alpha=1.0, linewidth=0.8)

        # Gradient Fill (Polygons)
        if len(nmu_valid) > 0:
            patches = []
            step = max(1, len(nmu_valid) // 300) # Optimization for PDF size

            for i in range(0, len(nmu_valid) - 1, step):
                idx_next = min(i + step, len(nmu_valid) - 1)
                x = [nmu_valid[i], nmu_valid[idx_next], nmu_valid[idx_next], nmu_valid[i]]
                y = [12, 12, 0, 0] # Fill vertical range
                polygon = Polygon(np.c_[x, y], closed=True)
                patches.append(polygon)

            collection = PatchCollection(patches, cmap=cm.viridis, alpha=0.25, zorder=1)
            collection.set_array(ks3_valid[::step][:len(patches)])
            collection.set_clim(0, np.pi)
            ax.add_collection(collection)

            # Solid fills outside the gradient region
            if boundary_val < 3.0:
                ax.axvspan(xmin=-3, xmax=-boundary_val, color=cmap(0), alpha=0.25)
                ax.axvspan(xmin=boundary_val, xmax=3, color=cmap(1.0), alpha=0.25)

    # --- FOREGROUND LINES (Common to ALL Deltas) ---

    # 1. The Gradient Curve (ks3) - NOW PLOTTED FOR ALL DELTAS
    if len(nmu_valid) > 0:
        segments = np.c_[nmu_valid[:-1], ne_k(nmu_valid[:-1], delta, ks3_valid[:-1]),
                         nmu_valid[1:], ne_k(nmu_valid[1:], delta, ks3_valid[1:])].reshape(-1, 2, 2)

        lc = LineCollection(segments, cmap=cm.viridis, linewidth=3, zorder=4) # Highest zorder
        lc.set_array(ks3_valid[:-1])
        lc.set_clim(0, np.pi)
        ax.add_collection(lc)

    # 2. Standard Lines (k=0, k=pi)
    # k=0 (Dark Purple)
    ax.plot(nmu, ne_k(nmu, delta, 0), color=cmap(0), linewidth=2.5, zorder=3)
    # k=pi (Yellow)
    ax.plot(nmu, ne_k(nmu, delta, np.pi), color=cmap(1.0), linewidth=2.5, zorder=3)

    # 3. Faint Intermediate lines
    for angle, alpha_val in [(np.pi/4, 0.5), (np.pi/2, 0.25), (3*np.pi/4, 0.5)]:
        ax.plot(nmu, ne_k(nmu, delta, angle), color=cmap(angle/np.pi),
                linewidth=1.5, zorder=2, alpha=alpha_val)

    # Limits and Tick Setup
    ax.set_xlim(-3, 3)
    ax.set_ylim(0, 10.0)
    ax.set_ylabel(r'$\Gamma_k/\omega$')
    ax.tick_params(axis='both', direction='in', which='both', top=True, right=True)

# 4. MAIN EXECUTION
def create_final_plot():
    # Reduced figsize to (6, 9) makes the font size (14pt) look larger relative to the plot
    fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True, gridspec_kw={'hspace': 0})

    nmu = np.linspace(-3, 3, 1000)
    deltas = [2.0, 0.5, 0.1]

    for i, ax in enumerate(axes):
        plot_band_panel(ax, deltas[i], nmu)
        if i == 2:
            ax.set_xlabel(r'$\mu/\omega$')

    # Colorbar positioning
    fig.subplots_adjust(right=0.85)
    cbar_ax = fig.add_axes([0.88, 0.11, 0.03, 0.77]) # [left, bottom, width, height]

    norm = Normalize(vmin=0, vmax=1.0)
    cb = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cm.viridis), cax=cbar_ax)
    # Use \pi explicitly in the label
    cb.set_label(r'$k/ \pi$', rotation=90, labelpad=20)

    plt.savefig('high_resolution_plot.png', bbox_inches='tight', dpi=600)
    plt.savefig('my_plot.pdf', bbox_inches='tight', format='pdf')
    plt.show()

create_final_plot()

: 